# 09 · Build the Semantic Model (Direct Lake on SQL)

Builds/updates the **`AzureCostAnalyzer_SM`** Direct Lake semantic model over the gold tables
produced by notebooks **01-08**, using **`semantic-link-labs` (`sempy_labs`)**: binds every table,
wires the relationships, marks the date table, and (re)creates all measures.

**Run this AFTER notebooks 01-08 have populated the Lakehouse** (`focus_partitioned`, the `gold_*`
tables, `dim_date`, `dim_month`, `dim_tag_key`).

**Idempotent** — reuses the existing model (same `itemId`) if present and only syncs its schema to
pick up new columns (e.g. the dynamic tag columns on `gold_chargeback_by_tag`); otherwise it creates
a new model. Relationships and measures are re-applied safely on every run.

**Model shape (matches the validated ACA app + report)**:
- **13 tables**: `focus_partitioned` (silver) + `gold_cost_summary_daily`, `gold_cost_summary_monthly`,
  `gold_cost_focus_monthly`, `gold_chargeback_by_tag` (WIDE dynamic tags), `dim_tag_key` (tag universe,
  no relationship), `gold_cost_by_resource`, `gold_reservations_coverage/waste/detail`,
  `gold_cost_anomalies`, `dim_date`, `dim_month`.
- **Relationships**: daily facts -> `dim_date[Date]`; monthly facts -> `dim_month[YearMonth]`;
  `dim_date -> dim_month`. `dim_tag_key` is bound but NOT related (the app EVALUATEs it to discover
  the dynamic tag columns).
- **Measures**: Cost, FOCUS usage-vs-rate, Time Intelligence, Anomalies, Reservations, Governance
  (via `TagCount`), Resource Detail, and Tags (on the WIDE `gold_chargeback_by_tag`).

In [ ]:
# Install semantic-link-labs FIRST. In Fabric, %pip can restart the Python session, which clears
# variables defined in earlier cells. Keeping it before the parameters ensures everything after it
# runs in the post-restart session.
%pip install semantic-link-labs --quiet

In [ ]:
# Run this notebook in the SAME workspace that holds the Lakehouse (sempy resolves both the
# lakehouse and the model by name in the current workspace — no default-lakehouse attach needed).
lakehouse_name = "AzureCostAnalyzer_LH"    # Lakehouse holding the gold_* tables
model_name     = "AzureCostAnalyzer_SM"    # semantic model to (re)create / update

## Bind the Direct Lake model (create or reuse)

Reuses the existing model (same `itemId`) and syncs its schema, or creates a new Direct-Lake-on-SQL
model binding all 13 tables. Forces a SQL-endpoint metadata sync first (Spark just rewrote the tables,
so the endpoint metadata can lag).

In [ ]:
import sempy_labs as labs
import sempy.fabric as fabric
from sempy_labs.tom import connect_semantic_model

# Schema-enabled Lakehouse: Spark writes tables under Tables/dbo/. sempy needs the source
# schema-qualified. Dict form keeps clean model table names (key) mapped to "dbo.<table>".
# `gold_chargeback_by_tag` is the WIDE dynamic tag table; `dim_tag_key` is the tag-universe helper
# the app reads to discover the (dynamic) tag columns -> bound with NO relationships.
MODEL_TABLES = ["focus_partitioned", "gold_cost_summary_daily", "gold_cost_summary_monthly",
                "gold_cost_focus_monthly", "gold_chargeback_by_tag", "dim_tag_key", "gold_cost_by_resource",
                "gold_reservations_coverage", "gold_reservations_waste", "gold_reservations_detail",
                "gold_cost_anomalies", "dim_date", "dim_month"]
TABLES = {t: f"dbo.{t}" for t in MODEL_TABLES}

# Direct-Lake-on-SQL reads through the Lakehouse SQL analytics endpoint. The tables were just
# (re)written by Spark, so the endpoint metadata can lag — force + wait for the sync.
print("Syncing SQL endpoint metadata (this can take a minute)...")
display(labs.refresh_sql_endpoint_metadata(item=lakehouse_name, type="Lakehouse"))

def _model_exists(name):
    try:
        df = fabric.list_datasets()
        col = next((c for c in df.columns if c.lower() in ("dataset name", "name")), df.columns[0])
        return name in set(df[col].astype(str))
    except Exception:
        return False

# Reuse the existing model if present (keeps the SAME itemId, so the app's fabric.yaml stays valid
# across re-runs). Only create a NEW model when it doesn't exist yet.
if _model_exists(model_name):
    print(f"Reusing existing model '{model_name}' (same itemId).")
    try:
        labs.directlake.direct_lake_schema_sync(dataset=model_name, add_to_model=True)
        print("Schema synced (added any new lakehouse columns to the model).")
    except Exception as e:
        print(f"  schema sync skipped: {e}")
else:
    print(f"Creating new model '{model_name}'.")
    labs.directlake.generate_direct_lake_semantic_model(
        dataset=model_name,
        tables=TABLES,
        source=lakehouse_name,
        source_type="Lakehouse",
        use_sql_endpoint=True,
        refresh=False,
    )

## Relationships + date table + measures

All idempotent: relationships are de-duplicated (a duplicate would fail with "ambiguous paths"), the
date table is marked, and each measure is dropped-then-re-added so re-runs pick up expression changes.

In [ ]:
CUR = "\\$#,##0"        # currency format string
CUR2 = "\\$#,##0.00"
PCT = "0.0%"
NUM = "#,##0"

with connect_semantic_model(dataset=model_name, readonly=False) as tom:

    # Snapshot existing relationships so re-runs on a REUSED model don't add duplicates.
    # (add_relationship does NOT error on a duplicate — it silently adds a second one,
    # which then fails at save time with "ambiguous paths between ...".)
    _existing_rels = {
        (r.FromTable.Name, r.FromColumn.Name, r.ToTable.Name, r.ToColumn.Name)
        for r in tom.model.Relationships
    }

    def rel(ft, fc, tt, tc):
        if (ft, fc, tt, tc) in _existing_rels:
            return
        tom.add_relationship(
            from_table=ft, from_column=fc, to_table=tt, to_column=tc,
            from_cardinality="Many", to_cardinality="One",
            cross_filtering_behavior="OneDirection", is_active=True,
        )
        _existing_rels.add((ft, fc, tt, tc))

    # Daily facts -> dim_date ; monthly facts -> dim_month ; dim_date -> dim_month
    rel("focus_partitioned",          "Date",      "dim_date",  "Date")
    rel("gold_cost_summary_daily",    "Date",      "dim_date",  "Date")
    rel("gold_cost_anomalies",        "Date",      "dim_date",  "Date")
    rel("gold_cost_summary_monthly",  "YearMonth", "dim_month", "YearMonth")
    rel("gold_cost_focus_monthly",    "YearMonth", "dim_month", "YearMonth")
    rel("gold_chargeback_by_tag",     "YearMonth", "dim_month", "YearMonth")
    rel("gold_cost_by_resource",      "YearMonth", "dim_month", "YearMonth")
    rel("gold_reservations_coverage", "YearMonth", "dim_month", "YearMonth")
    rel("gold_reservations_waste",    "YearMonth", "dim_month", "YearMonth")
    rel("gold_reservations_detail",   "YearMonth", "dim_month", "YearMonth")
    rel("dim_date",                   "YearMonth", "dim_month", "YearMonth")

    try:
        tom.mark_as_date_table(table_name="dim_date", column_name="Date")
    except Exception:
        pass

    def m(tbl, name, expr, fmt=None, folder=None):
        # idempotent: drop any existing measure of the same name before (re)adding,
        # so re-runs against a reused model pick up expression/format changes.
        try:
            t = tom.model.Tables[tbl]
            for ms in list(t.Measures):
                if ms.Name == name:
                    t.Measures.Remove(ms)
        except Exception:
            pass
        tom.add_measure(table_name=tbl, measure_name=name, expression=expr,
                        format_string=fmt, display_folder=folder)

    # -- Cost --
    m("gold_cost_summary_monthly", "Total Effective Cost", "SUM('gold_cost_summary_monthly'[EffectiveCost])", CUR, "Cost")
    m("gold_cost_summary_monthly", "Total Billed Cost",    "SUM('gold_cost_summary_monthly'[BilledCost])",    CUR, "Cost")
    m("gold_cost_summary_monthly", "Total List Cost",      "SUM('gold_cost_summary_monthly'[ListCost])",      CUR, "Cost")
    m("gold_cost_summary_monthly", "Total Savings",        "SUM('gold_cost_summary_monthly'[SavingsAmount])", CUR, "Cost")
    m("gold_cost_summary_monthly", "Savings %",            "DIVIDE([Total Savings],[Total List Cost])",       PCT, "Cost")
    m("gold_cost_focus_monthly",   "Real Savings",         "SUM('gold_cost_focus_monthly'[SavingsAmount])",   CUR, "Cost")
    m("gold_cost_focus_monthly",   "Total Baseline Cost",  "SUM('gold_cost_focus_monthly'[SavingsBaseline])", CUR, "Cost")

    # -- FOCUS usage-vs-rate (home: focus_partitioned) — powers Explorer bridge + anomaly window --
    m("focus_partitioned", "FX Effective Cost",    "SUM('focus_partitioned'[EffectiveCost])",    CUR2, "FOCUS")
    m("focus_partitioned", "FX Consumed Quantity", "SUM('focus_partitioned'[ConsumedQuantity])", "#,##0.0", "FOCUS")

    # -- Time Intelligence (monthly facts relate via dim_month; use MonthStart for MoM) --
    m("gold_cost_summary_monthly", "Last Month Cost",
      "VAR d = MAX('dim_month'[MonthStart]) "
      "RETURN CALCULATE([Total Effective Cost], REMOVEFILTERS('dim_month'), 'dim_month'[MonthStart] = EDATE(d, -1))",
      CUR, "Time Intelligence")
    m("gold_cost_summary_monthly", "MoM Cost %",
      "DIVIDE([Total Effective Cost] - [Last Month Cost], [Last Month Cost])", PCT, "Time Intelligence")

    # -- Anomalies --
    m("gold_cost_anomalies", "Anomaly Count",
      "CALCULATE(COUNTROWS('gold_cost_anomalies'), 'gold_cost_anomalies'[IsAnomaly] = TRUE())", NUM, "Anomalies")
    m("gold_cost_anomalies", "Anomaly Count (KPI)", "[Anomaly Count]", NUM, "Anomalies")

    # -- Reservations --
    m("gold_reservations_coverage", "RI Coverage %",
      "DIVIDE(SUM('gold_reservations_coverage'[ReservedCost]), SUM('gold_reservations_coverage'[TotalCost]))", PCT, "Reservations")
    m("gold_reservations_waste",    "RI Waste Cost", "SUM('gold_reservations_waste'[WasteCost])", CUR, "Reservations")
    m("gold_cost_focus_monthly",    "Net Reservation Savings",
      "CALCULATE(SUM('gold_cost_focus_monthly'[SavingsAmount]), 'gold_cost_focus_monthly'[PricingCategory] = \"Committed\")", CUR, "Reservations")
    m("gold_reservations_waste",    "Used Reservation Savings", "SUM('gold_reservations_detail'[SavingsUsed])", CUR, "Reservations")
    m("gold_reservations_detail",   "Used Reservation Cost",    "SUM('gold_reservations_detail'[UsedCost])",   CUR, "Reservations")
    m("gold_reservations_detail",   "Wasted Reservation Cost",  "SUM('gold_reservations_detail'[WasteCost])",  CUR, "Reservations")
    m("gold_reservations_detail",   "Reservation Utilization %",
      "DIVIDE(SUM('gold_reservations_detail'[UsedCost]), SUM('gold_reservations_detail'[UsedCost]) + SUM('gold_reservations_detail'[WasteCost]))", PCT, "Reservations")

    # -- Governance (home: gold_cost_by_resource; TagCount = 0 means fully untagged) --
    m("gold_cost_by_resource", "Untagged Cost",
      "CALCULATE(SUM('gold_cost_by_resource'[EffectiveCost]), 'gold_cost_by_resource'[TagCount] = 0)", CUR, "Governance")
    m("gold_cost_by_resource", "Untagged %",
      "DIVIDE([Untagged Cost], SUM('gold_cost_by_resource'[EffectiveCost]))", PCT, "Governance")
    m("gold_cost_by_resource", "Untagged Resource Count",
      "CALCULATE(DISTINCTCOUNT('gold_cost_by_resource'[ResourceId]), 'gold_cost_by_resource'[TagCount] = 0)", NUM, "Governance")
    m("gold_cost_by_resource", "Tag Count", "SUM('gold_cost_by_resource'[TagCount])", NUM, "Governance")
    m("gold_cost_by_resource", "Tags In Filter",
      "CALCULATE(DISTINCTCOUNT('gold_cost_by_resource'[ResourceId]), 'gold_cost_by_resource'[TagCount] > 0)", NUM, "Governance")

    # -- Resource Detail --
    m("gold_cost_by_resource", "Resource Effective Cost", "SUM('gold_cost_by_resource'[EffectiveCost])", CUR, "Resource Detail")
    m("gold_cost_by_resource", "Resource Count", "DISTINCTCOUNT('gold_cost_by_resource'[ResourceId])", NUM, "Resource Detail")

    # -- Tags (home: gold_chargeback_by_tag WIDE; grouped by any tag COLUMN discovered via dim_tag_key) --
    # Cost is exact at ONE row per resource-month, so multi-tag grouping/filtering does NOT double count.
    m("gold_chargeback_by_tag", "Tag Effective Cost", "SUM('gold_chargeback_by_tag'[EffectiveCost])", CUR, "Tags")
    m("gold_chargeback_by_tag", "Tagged Resource Count",
      "CALCULATE(DISTINCTCOUNT('gold_chargeback_by_tag'[ResourceId]), 'gold_chargeback_by_tag'[TagCount] > 0)", NUM, "Tags")
    m("gold_chargeback_by_tag", "Untagged Resource Count (Tags)",
      "CALCULATE(DISTINCTCOUNT('gold_chargeback_by_tag'[ResourceId]), 'gold_chargeback_by_tag'[TagCount] = 0)", NUM, "Tags")

## Refresh (reframe Direct Lake) + validate

In [ ]:
import sempy.fabric as fabric

# Reframe Direct Lake (pick up the freshly written Delta data).
labs.refresh_semantic_model(dataset=model_name)

# Definitive validation: run a DAX query against the model. If this returns a value, the model +
# measures + relationships all work end-to-end.
print("Sanity check (DAX against the model):")
try:
    display(fabric.evaluate_dax(
        dataset=model_name,
        dax_string='EVALUATE ROW("Total Effective Cost", [Total Effective Cost], "Untagged %", [Untagged %], "Tag Effective Cost", [Tag Effective Cost])',
    ))
except Exception as e:
    print(f"  sanity DAX skipped: {e}")

# Tag universe the app will discover at runtime.
print("Tag universe (dim_tag_key):")
try:
    display(fabric.evaluate_dax(dataset=model_name, dax_string='EVALUATE dim_tag_key ORDER BY dim_tag_key[Rank]'))
except Exception as e:
    print(f"  dim_tag_key query skipped: {e}")

# Best-effort measure listing (some sempy/TOM versions raise on Direct Lake models).
print("Measures:")
try:
    display(fabric.list_measures(dataset=model_name))
except Exception as e:
    print(f"  list_measures skipped: {e}")

print(f"\nDone. Point the app at model '{model_name}' (set workspaceId + itemId in app/fabric.yaml).")

## (optional) App connection IDs

Prints the `workspaceId` + `itemId` of the model, ready to paste into the app's `app/fabric.yaml`
under `semanticModels.aca` so the app queries this model.

In [ ]:
import sempy.fabric as fabric

ws_id = fabric.get_workspace_id()
try:
    item_id = fabric.resolve_item_id(item_name=model_name, type="SemanticModel")
except Exception:
    ds = fabric.list_datasets()
    name_col = next(c for c in ds.columns if c.lower() in ("dataset name", "name"))
    id_col   = next(c for c in ds.columns if c.lower() in ("dataset id", "id"))
    item_id = ds.loc[ds[name_col] == model_name, id_col].iloc[0]

print(f"aca workspaceId: {ws_id}")
print(f"aca itemId:      {item_id}")
print("\n--- paste into app/fabric.yaml (semanticModels.aca) ---")
print(f"        workspaceId: {ws_id}")
print(f"        itemId: {item_id}")